<a href="https://colab.research.google.com/github/rombird/AI-bigdata-course/blob/main/m04/day3_%EB%85%B9%EC%9D%8C%EB%90%9C_%ED%8C%8C%EC%9D%BC_%EC%A0%95%EB%A6%AC%ED%95%B4%EC%A3%BC%EB%8A%94_AI_%EC%84%9C%EA%B8%B0(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
HUGGING_FACE_TOKEN = userdata.get('HUGGING_FACE_TOKEN')

In [ ]:
%pip install python-dotenv
%pip install OpenAI

In [ ]:
!wget https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/lsy_audio_2023_58s.mp3

--2026-06-25 12:57:18--  https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/lsy_audio_2023_58s.mp3
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 928958 (907K) [audio/mpeg]
Saving to: ‘lsy_audio_2023_58s.mp3’

lsy_audio_2023_58s. 100%[===================>] 907.19K  --.-KB/s    in 0.05s   

2026-06-25 12:57:18 (19.6 MB/s) - ‘lsy_audio_2023_58s.mp3’ saved [928958/928958]



In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [ ]:
# mp3 음성을 텍스트로 변환


# mp3 파일 경로 설정
audio_file_path='/content/lsy_audio_2023_58s.mp3'

# mp3 음성 파일을 텍스트로 변환한 결과(파일을 읽어야 한다)
# 파일 읽을 때 -> r / 텍스트 -> t(생략가능) / 그림, 소리, 동영상 등 (텍스트 X) -> b
with open(audio_file_path, 'rb') as audio_file:
  transcription = client.audio.transcriptions.create(
      model='whisper-1',
      file=audio_file
  )

transcription

Transcription(text='안녕하세요. 이 강의는 GPT API로 챗봇 만들기라는 내용을 다루는 강의입니다. GPT API에 대해서 생소하신 분들도 있을 텐데 우리가 잘 알고 있는 채GPT, 채GPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 여러 가지들이 있는데 좀 이 강의의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챗봇 프로그램을 만드는 게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러 가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고', logprobs=None, usage=UsageDuration(seconds=58.0, type='duration'))

In [ ]:
# 한국어 음성을 영어로 번역하기
with open(audio_file_path, 'rb') as audio_file:
  translation = client.audio.translations.create(
      model='whisper-1',
      file=audio_file
  )
translation

Translation(text="Hello, this is a lecture on how to make a chatbot with GPT API. Some of you may be unfamiliar with GPT API. We're going to talk about how to make the program we want using the chat GPT function that we know well. So there are a lot of lectures like this. There are many things, but if I were to say the characteristics of this lecture, it's not easy to make a chatbot program that achieves a clear mission with GPT. I'm going to talk about how to implement this and why it's necessary. As an example, there can be many examples. The example here is to create a program that automatically creates a music playlist video through conversation. So let me show you how the program runs. The program we're going to make is going to look like this.")

In [ ]:
!pip list

Package                                  Version
---------------------------------------- ------------------
absl-py                                  1.4.0
accelerate                               1.14.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.2
aiohttp                                  3.14.1
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.12.0
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                          

In [ ]:
# 허깅페이스에서 제공하는 코드 복사 -> 일부 수정
# 허깅페이스 'whisper-large-v3' 검색
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer, # 토크나이저 : 텍스트를 작은 단위로 쪼개는 도구
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s = 2,
)

sample = '/content/lsy_audio_2023_58s.mp3'
result = pipe(sample)
print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[tran

{'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다. GPT-API에 대해서 생소하신 분들도 있을텐데 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챕터 프로그램을 만드는게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러가지가 될 수 있는데 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고', 'chunks': [{'timestamp': (0.0, 6.3), 'text': ' 안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.'}, {'timestamp': (7.18, 10.0), 'text': ' GPT-API에 대해서 생소하신 분들도 있을텐데'}, {'timestamp': (11.0, 17.0), 'text': ' 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서'}, {'timestamp': (17.0, 20.0), 'text': ' 우리가 원하는 프로그램을 어떻게 만드는지에 대해서'}, {'timestamp': (20.0, 22.0), 'text': ' 이야기할 거예요.'}, {'timestamp': (22.0, 24.0), 'text': ' 그래서 이런 강의들이 사실 많이 있습니다.'}, {'timestamp': (24.0, 27.48), 'text': ' 그래서 여러가지들이 있는데 이 강의 특징이라고 한다면'}, {'timestamp': (27.48, 29.58), 'text': ' G

In [ ]:
# chunks(청크)를 csv파일로 저장하기
# 청크 : 덩어리, 컴퓨터 공학에서는 일정한 크기의 저장공간

# {'chunks':[{'timestamp'} : (0.0, 6.3), 'text':'']} 형태로 chunks부분 출력
start_end_text = []
for chunk in result['chunks']:
  start = chunk['timestamp'][0] # 시작 시각
  end = chunk['timestamp'][1] # 끝 시각
  text = chunk['text'] # 내용
  start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=['start', 'end', 'text']) # 데이터프레임으로 생성
df.to_csv('audio.csv', index=False, sep='|') # 데이터프레임을 csv 파일로 내보내기
display(df)

,start,end,text
0,0.00,6.30,안녕하세요. 이 강의는 GPT-API로 챗봇 만들기 라는 내용을 다루는 강의입니다.
1,7.18,10.00,GPT-API에 대해서 생소하신 분들도 있을텐데
2,11.00,17.00,"우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서"
3,17.00,20.00,우리가 원하는 프로그램을 어떻게 만드는지에 대해서
4,20.00,22.00,이야기할 거예요.
5,22.00,24.00,그래서 이런 강의들이 사실 많이 있습니다.
6,24.00,27.48,그래서 여러가지들이 있는데 이 강의 특징이라고 한다면
7,27.48,29.58,GPT로 명확한 미션을 달성하는
8,29.58,31.66,챕터 프로그램을 만드는게 사실
9,31.66,34.32,쉽지는 않은데 이걸 어떻게 해서


In [ ]:
%pip install pyannote.audio
%pip install numpy==1.26

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.5/893.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 86.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [ ]:
from pyannote.audio import Pipeline

# 파이프라인 --> 여러처리단계를 연결한 작업흐름
pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  token=HUGGING_FACE_TOKEN)

config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

위 코드가 에러나면 아래 onnxruntime-gpu 해보기 → 다시 위 코드 실행



In [ ]:
%pip install onnxruntime-gpu==1.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 4.2 MB/s eta 0:00:00


In [ ]:
# # 1. 기존에 설치된 꼬인 버전을 지웁니다.
# %pip uninstall -y onnxruntime onnxruntime-gpu

# # 2. 코랩 환경과 호환이 잘 되는 버전으로 지정하여 설치합니다.
# %pip install onnxruntime-gpu==1.18.0

# 3. 그 다음 import 확인
import onnxruntime
print("onnxruntime 설치 완료, 버전:", onnxruntime.__version__)

onnxruntime 설치 완료, 버전: 1.18.0


In [ ]:
import torch

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))
    print('cuda is available')
else:
    print('cuda is not available')

cuda is available


In [ ]:
!wget https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/싼기타_비싼기타.mp3

--2026-06-25 13:02:39--  https://raw.githubusercontent.com/saintdragon2/gpt_agent_2025_easyspub/main/chap05/audio/%EC%8B%BC%EA%B8%B0%ED%83%80_%EB%B9%84%EC%8B%BC%EA%B8%B0%ED%83%80.mp3
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6968467 (6.6M) [audio/mpeg]
Saving to: ‘싼기타_비싼기타.mp3’

싼기타_비싼기타.mp3 100%[===================>]   6.65M  --.-KB/s    in 0.1s    

2026-06-25 13:02:39 (69.1 MB/s) - ‘싼기타_비싼기타.mp3’ saved [6968467/6968467]



In [ ]:
# 기존 코드 → 버전 이슈로 실행 X
diarization = pipeline("/content/싼기타_비싼기타.mp3")

with open("/content/싼기타_비싼기타.rttm", "w") as rttm:
    diarization.write_rttm(rttm)

/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


ValueError: requested chunk [ 00:00:00.000 -->  00:00:10.000] from 싼기타_비싼기타 file resulted in 439895 samples instead of the expected 441000 samples.

In [ ]:
# ffmpeg로 MP3 → WAV 변환 (16kHz 모노로)
!ffmpeg -i "/content/싼기타_비싼기타.mp3" -ar 16000 -ac 1 "/content/싼기타_비싼기타.wav" -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# WAV 파일로 파이프라인 실행
# diarization = pipeline("/content/싼기타_비싼기타.wav")

# with open("/content/싼기타_비싼기타.rttm", "w") as rttm:
#     diarization.write_rttm(rttm)

# diarization.speaker_diarization 이 실제 Annotation 객체
annotation = diarization.speaker_diarization

with open("/content/싼기타_비싼기타.rttm", "w") as rttm:
    annotation.write_rttm(rttm)

In [ ]:
# AttributeError: 'DiarizeOutput' object has no attribute 'write_rttm'
# -> DiarizeOutput 뭔지 확인해보기
print(type(diarization))
print(dir(diarization))

# DiarizeOutput은 dataclass이고, 실제 diarization 결과는 .speaker_diarization 속성 안에 있어요.

<class 'pyannote.audio.pipelines.speaker_diarization.DiarizeOutput'>
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'exclusive_speaker_diarization', 'serialize', 'speaker_diarization', 'speaker_embeddings']


In [ ]:
import pyannote.audio
print(pyannote.audio.__version__)

4.0.5
